### Парсинг пресс-релизов

Сбор данных осуществляется через API сайта ЦБ с последовательным обходом страниц. Для каждого пресс-релиза загружаются дата публикации, заголовок, ссылка и полный текст документа. Далее HTML-разметка очищается от тегов и лишних пробелов, текст приводится к единому формату.

In [17]:
%pip install --quiet --disable-pip-version-check pandas==3.1.0 beautifulsoup4==4.15.0 workalendar curl-cffi

In [11]:
import json
import time
from datetime import date
from bs4 import BeautifulSoup
from curl_cffi import requests
import re
import pandas as pd
import numpy as np

INLINE_TAGS = ["strong", "b", "i", "em", "u", "span", "a", "sup", "sub", "mark"]
BLOCK_TAGS = ["p", "div", "li", "tr", "h1", "h2", "h3", "h4", "h5", "h6", "blockquote"]

BASE = "https://www.cbr.ru"
IMPERSONATE = "chrome124"
TIMEOUT = 10
RETRIES = 3
SLEEP_BETWEEN_REQUESTS = 2.0
SLEEP_BETWEEN_RELEASES = 7.0

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,"
        "image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Upgrade-Insecure-Requests": "1",
}


def _get(url, **kwargs):
    """GET с ретраями и таймаутом."""
    for attempt in range(RETRIES):
        try:
            res = requests.get(
                url,
                impersonate=IMPERSONATE,
                timeout=TIMEOUT,
                headers=HEADERS,
                default_headers=True,
                **kwargs,
            )
            res.raise_for_status()
            return res
        except Exception as e:
            if attempt == RETRIES - 1:
                raise
            print(f"Попытка {attempt + 1} не удалась: {e}")
            time.sleep(SLEEP_BETWEEN_REQUESTS * (attempt + 1))

def get_press_releases(page=0):
    """Получает список пресс-релизов по ключевой ставке через API ЦБ."""
    url = f"{BASE}/news/eventandpress/"
    params = {
        "page": page,
        "IsEng": "false",
        "type": 4,  # тип события: пресс-релизы
        "dateFrom": "1991-01-23T00:00:00",
        "dateTo": date.today().isoformat(),  # динамическая дата
        "Tid": 0,
        "vol": "KEYRATE",  # тема: ключевая ставка
        "phrase": "",
        "pagesize": 40,
    }

    res = _get(url, params=params)
    content_type = res.headers.get("Content-Type", "")
    if "application/json" not in content_type:
        raise ValueError(f"Unexpected content-type: {content_type}")

    data = res.json()

    # Поддержка двух форматов ответа: {"items": [...], "total": N} или плоский список
    if isinstance(data, dict):
        items = data.get("items", [])
        total = data.get("total", 0)
        return items, total
    if isinstance(data, list):
        return data, len(data)

    raise ValueError(f"Unknown response structure: {type(data)}")


def press_release_url(file_name):
    """Собирает URL страницы пресс-релиза по имени файла."""
    return f"{BASE}/press/pr/?file={file_name}"


def parse_press_page(file_name):
    url = press_release_url(file_name)
    res = _get(url)
    soup = BeautifulSoup(res.text, "html.parser")
    body = soup.select_one(".news_text, .content, #content")
    if body is None:
        raise ValueError(f"Content block not found for {file_name}")

    for tag in body.find_all(INLINE_TAGS):
        tag.unwrap()

    for br in body.find_all("br"):
        br.replace_with(" ")

    for block in body.find_all(BLOCK_TAGS):
        block.insert_after(" ")

    text = body.get_text()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def collect_all_releases(sleep=SLEEP_BETWEEN_RELEASES):
    """Собирает все пресс-релизы, идя по страницам, пока не встретит пустую."""
    all_releases = []
    seen_files = set()
    page = 0

    while True:
        items, _total = get_press_releases(page=page)

        if not items:
            print(f"Страница {page}: пусто, завершаю")
            break

        print(f"Страница {page}: {len(items)} записей")

        for item in items:
            file_name = item["doc_htm"]
            if file_name in seen_files:
                continue
            seen_files.add(file_name)

            release = {
                "title": item["name_doc"].replace("\xa0", " "),
                "date": item["dateupdate"],
                "url": press_release_url(file_name),
            }

            try:
                release["text"] = parse_press_page(file_name)
                release["parse_status"] = "success"

            except Exception as e:
                print(f"Не удалось загрузить {file_name}: {e}")
                release["text"] = None
                release["parse_status"] = "error"

            all_releases.append(release)
            time.sleep(sleep)

        page += 1

    print(f"\nВсего собрано: {len(all_releases)}")
    return all_releases


In [8]:
releases = collect_all_releases()

df = pd.DataFrame(releases)

Страница 0: 20 записей
Страница 1: 20 записей
Страница 2: 20 записей
Страница 3: 20 записей
Страница 4: 20 записей
Страница 5: 17 записей
Страница 6: пусто, завершаю

Всего собрано: 117


In [26]:
df[df['parse_status']=='success']
df.drop(columns=['parse_status'],inplace=True)

### Предобработка

Важно различать два понятия: публикация, посвящённая ключевой ставке, и пресс-релиз с решением по ключевой ставке. Фильтр API по теме не гарантирует, что каждая запись содержит решение по ставке; поэтому релевантность дополнительно проверяется по заголовку и тексту.
На данном этапе удаляются публикации, которые попали в выдачу по теме ключевой ставки, но не описывают решение по ключевой ставке.
Далее из текста удаляются элементы интерфейса сайта и повторяющиеся служебные фрагменты (например, блоки «Поделиться» и «Сохранить в PDF»).


In [27]:
df["date"] = pd.to_datetime(df["date"], errors="coerce", format="mixed")

# Убираем 5 нерелевантных публикаций по title + date
exclude_pairs = [
    ("2015-12-29", "О фиксированном времени публикации ключевой информации по статистике внешнего сектора"),
    ("2015-09-11", "О публикации ежемесячной оценки ключевых агрегатов платежного баланса Российской Федерации"),
    ("2015-05-06", "О публикации Обзора ключевых показателей некредитных финансовых организаций за 2014 год"),
    ("2013-09-13", "О системе процентных инструментов денежно-кредитной политики Банка России"),
    ("2013-09-13", "О процентных ставках по операциям Банка России"),
]
df["title_norm"] = (
    df["title"].fillna("")
    .str.replace("\u00a0", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
date_norm = df["date"].dt.normalize()
mask_exclude = pd.Series(False, index=df.index)
for d, t in exclude_pairs:
    mask_exclude |= date_norm.eq(pd.Timestamp(d)) & df["title_norm"].eq(t)
df_clean = df.loc[~mask_exclude].copy().reset_index(drop=True)
print("Исходно:", len(df), "| исключено:", int(mask_exclude.sum()), "| осталось:", len(df_clean))


Исходно: 117 | исключено: 5 | осталось: 112


In [28]:
CUT_MARKERS = [
    r"При использовании материала",
    r"Процентные ставки по ",
    r"Сохранить в PDF",
    r"Заявление Председателя Банка России",
]

def strip_boilerplate(s: str) -> str:
    if not isinstance(s, str):
        return ""
    # префикс + share-блок
    s = re.sub(r"^\s*Новости\s+", "", s, flags=re.I)
    s = re.sub(r"\bПресс-релиз\b", " ", s)
    s = re.sub(r"Поделиться\s+ВКонтакте\s+Telegram\s+Макс", " ", s)

    # обрезаем по самому раннему boilerplate-маркеру
    cut = len(s)
    for pat in CUT_MARKERS:
        m = re.search(pat, s)
        if m and m.start() < cut:
            cut = m.start()
    s = s[:cut]

    # нормализация
    s = re.sub(r"\s+", " ", s).strip()
    return s

df_clean["text_clean"] = df_clean["text"].apply(strip_boilerplate)

### Формирование таргета

Данные о ключевой ставке загружаются с сайта Банка России и очищаются, даты приводятся к единому формату, значения ставок к числовому типу. Для каждого пресс-релиза определяется дата вступления ставки в силу с учётом выходных и праздничных дней. Затем к каждому заседанию привязывается соответствующее значение ключевой ставки. На основе текущей и следующей ставки формируется целевая переменная target_next, отражающая направление изменения ставки на следующем заседании:
- -1 — снижение ключевой ставки;
- 0 — сохранение ставки без изменений;
- 1 — повышение ключевой ставки.


In [33]:
# Загружаем ставки ЦБ

url = (
    "https://www.cbr.ru/hd_base/KeyRate/"
    "?UniDbQuery.Posted=True"
    "&UniDbQuery.From=17.09.2013"
    f"&UniDbQuery.To={date.today().strftime('%d.%m.%Y')}"
)

tables = pd.read_html(url, decimal=",", thousands=".")
df_rates = tables[0].copy()

date_str = (
    df_rates["Дата"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
    .str.zfill(8)
)

df_rates["Дата"] = pd.to_datetime(date_str,format="%d%m%Y",errors="coerce")

df_rates["Ставка"] = pd.to_numeric(df_rates["Ставка"],errors="coerce")

df_rates = (
    df_rates
    .rename(columns={"Дата": "rate_date", "Ставка": "rate"})
    .dropna(subset=["rate_date", "rate"])
    .sort_values("rate_date")
    .drop_duplicates(subset=["rate_date"], keep="last")
    .reset_index(drop=True)
)

df_rates["rate_date"] = df_rates["rate_date"].dt.normalize()



# Функция проверки рабочего дня
from workalendar.europe import Russia
workcalendar = Russia()

def is_workday(d):
    if 2015 <= d.year <= 2027:
        return workcalendar.is_working_day(d.date())

    return d.dayofweek < 5

# Ищем дату вступления в силу ставки для каждого заседания
def get_target_date(meeting_date):
    meeting_date = pd.Timestamp(meeting_date).normalize()
    # Ближайший понедельник после заседания
    days_until_monday = (7 - meeting_date.dayofweek) % 7
    if days_until_monday == 0:
        days_until_monday = 7
    monday = meeting_date + pd.Timedelta(days=days_until_monday)
    # Если понедельник рабочий, используем его
    if is_workday(monday):
        return monday
    # Если понедельник выходной, ищем ближайший рабочий день
    candidates = [
        monday + pd.Timedelta(days=offset)
        for offset in [-3, -2, -1, 1, 2, 3]
    ]

    candidates = [
        d for d in candidates
        if d > meeting_date
        and d.dayofweek != 0
        and is_workday(d)
    ]

    if not candidates:
        return monday

    return min(candidates, key=lambda d: abs((d - monday).days))


def get_rate(target_date):
    target_date = pd.Timestamp(target_date).normalize()

    # Ищем ставку на целевую дату или после неё
    future_rates = df_rates.loc[
        df_rates["rate_date"] >= target_date
    ].sort_values("rate_date")

    if future_rates.empty:
        return float("nan")

    return future_rates.iloc[0]["rate"]

df_clean["date"] = pd.to_datetime(df_clean["date"]).dt.normalize()
df_clean["rate_date"] = df_clean["date"].apply(get_target_date)

df_clean_w_target = df_clean.copy()
df_clean_w_target["rate_current"] = (df_clean_w_target["rate_date"].apply(get_rate))
df_clean_w_target = (df_clean_w_target.sort_values("date").reset_index(drop=True))

# Ставка после следующего заседания
df_clean_w_target["rate_next"] = (df_clean_w_target["rate_current"].shift(-1))

# Направление изменения ставки на следующем заседании
df_clean_w_target["target_next"] = np.sign(
    df_clean_w_target["rate_next"] -
    df_clean_w_target["rate_current"])

# # Последняя строка не имеет следующего заседания
df_clean_w_target = df_clean_w_target.dropna(subset=["rate_current", "rate_next"])

df_clean_w_target["target_next"] = (df_clean_w_target["target_next"].astype(int))


df_clean_w_target = df_clean_w_target.sort_values("date").reset_index(drop=True)
df_clean_w_target["year"] = df_clean_w_target["date"].dt.year
df_clean_w_target["month"] = df_clean_w_target["date"].dt.month
df_clean_w_target["weekday"] = df_clean_w_target["date"].dt.dayofweek
df_clean_w_target["len_chars"] = df_clean_w_target["text_clean"].str.len()
df_clean_w_target["len_words"] = df_clean_w_target["text_clean"].str.split().apply(len)
df_clean_w_target["days_since_prev"] = (df_clean_w_target["date"].diff().dt.days)
df_clean_w_target=df_clean_w_target[['date','url','title_norm','text_clean','year','month','weekday','len_chars',
                                     'len_words','days_since_prev','rate_current','rate_next','target_next']]

In [ ]:
df_clean_w_target.to_csv('keyrate_press_releases.csv')